In [29]:
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.animation import FuncAnimation
import h5py

In [30]:

fname1 = 'SPU_test2026-07-07_15-06-30.h5'
# fname1 = '2026-06-15/SPU_OS_eScan_10.0keV_h5_2026-06-15_22-09-29.h5'
# fname2 = '2026-06-15/SPU_CS_eScan_10.0keV_h5_2026-06-15_22-17-32.h5'


In [31]:

def read_data(fname):
    
    energy = []
    flux = []
    exp_time = []
    images = {} 
    max_intensity = []
    max_intensity_global = 0

    with h5py.File(fname, 'r') as f:
        group = f['images']
        keys = list(group.keys())

        for i in range(len(keys)):
            energy.append(group[keys[i]].attrs['energy_mon'])
            flux.append(group[keys[i]].attrs['flux_mon'])
            exp_time.append([group[keys[i]].attrs['exp_time_mon'],
                            group[keys[i]].attrs['exp_time_sp']])
            images[keys[i]] = np.array(group[keys[i]])
            max_intensity.append(group[keys[i]].attrs['max_intensity'])
            max_intensity_global = np.max([max_intensity_global, 
                                    group[keys[i]].attrs['max_intensity']])

    print('global max. intensity = {:0d}/1024 counts'.format(max_intensity_global))

    energy = np.array(energy)
    exp_time = np.array(exp_time)
    flux = np.array(flux) / exp_time[:,0]
    # flux /= np.max(flux)
    max_intensity = np.array(max_intensity)
    
    return energy, flux, images


In [32]:
energy1, flux1, images1 = read_data(fname1)
# energy2, flux2, images2 = read_data(fname2)

global max. intensity = 14/1024 counts


In [33]:
energy1

array([ 8.99997044,  9.17492676,  9.34993553,  9.52490139,  9.70004177,
        9.70000553,  9.71181393,  9.72401714,  9.73586655,  9.74799728,
        9.76000595,  9.77201653,  9.78405857,  9.79585743,  9.80805969,
        9.81987095,  9.83208847,  9.84391403,  9.85583305,  9.86806774,
        9.88009262,  9.89195347,  9.90386009,  9.91588116,  9.92803478,
        9.93999386,  9.95182228,  9.96396255,  9.97607613,  9.98802185,
       10.0000906 , 10.01187515, 10.02387905, 10.03604603, 10.04794598,
       10.06008625, 10.0720253 , 10.08407021, 10.09599018, 10.10806561,
       10.12001324, 10.13178635, 10.14398766, 10.15605164, 10.16785049,
       10.17983627, 10.19201756, 10.20407009, 10.21587372, 10.22788429,
       10.23991489, 10.25187397, 10.26408195, 10.27607822, 10.28790092,
       10.30010605, 10.39997768, 10.52016068, 10.64010143, 10.75987053,
       10.88006687, 10.99989986, 11.11999416, 11.23997498, 11.36001682,
       11.47998619, 11.60016632, 11.70006561, 11.71488571, 11.73

Animated Plot

In [34]:
%matplotlib qt5

energy = energy1
flux = flux1
images = images1

# Setup figure with two equal-sized panels
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(9, 4), 
                               gridspec_kw={'width_ratios': [1, 1]})

line, = ax1.plot([], [], '.-k')
ax1.set_xlim(energy.min(), energy.max())
ax1.set_ylim(0, np.max(flux))
ax1.set_title("Flux vs Energy")
ax1.set_xlabel('Energy [keV]')
ax1.set_ylabel('Flux [a.u.]')

# Initialize the image display with the first image
first_img = images['img_001']
im = ax2.imshow(first_img, cmap='viridis', origin='lower', vmin=0, vmax=999)
ax2.set_title("Image")
ax2.set_xlabel('X [px]')
ax2.set_ylabel('Y [px]')

# Add colorbar with same height as ax2
cbar = fig.colorbar(im, ax=ax2, fraction=0.046, pad=0.04)
cbar.set_label("Intensity [a.u.]")

# Initialization function
def init():
    line.set_data([], [])
    im.set_data(np.zeros_like(first_img))
    return line, im

# Update function for animation
def update(frame):
    line.set_data(energy[:frame], flux[:frame])
    key = 'img_{0:03d}'.format(frame + 1)
    if key in images:
        im.set_data(images[key])
    return line, im

# Create animation
ani = FuncAnimation(fig, update, frames=len(energy) + 1,
                    init_func=init, blit=False, interval=300)

fig.tight_layout()

# ani.save("spectrum_measurement.gif", writer="ffmpeg", fps=5, dpi=150)




PLOT FLUX ONLY

In [35]:

fig, ax = plt.subplots()
ax.plot(energy, flux, 'o-')
ax.set_xlabel('Energy [keV]')
ax.set_ylabel('Flux [a.u.]')
ax.grid(alpha=0.3)
ax.minorticks_on()
ax.tick_params(which='both', axis='both', direction='in', top=True, right=True)
plt.show()

PLOT BOTH CURVES

In [59]:
fig, ax = plt.subplots(figsize=(4.5, 3.0))
ax.plot(energy1, flux1, '-', label='Open Slits')
ax.plot(energy2, flux2, '-', label='Closed Slits')
ax.set_xlabel('Energy [keV]')
ax.set_ylabel('Flux [a.u.]')
ax.grid(alpha=0.3)
ax.legend()
# ax.minorticks_on()
ax.tick_params(which='both', axis='both', direction='in', top=True, right=True)
fig.tight_layout()
plt.savefig('spectrum_slit_comparison.png', dpi=300)
plt.show()

In [58]:

flux1_aux = flux1-np.min(flux1)
flux2_aux = flux2-np.min(flux2)
fig, ax = plt.subplots(figsize=(4.5, 3.0))
ax.plot(energy1, flux1_aux/np.max(flux1_aux), '-', label='Open Slits')
ax.plot(energy2, flux2_aux/np.max(flux2_aux), '-', label='Closed Slits')
ax.set_xlabel('Energy [keV]')
ax.set_ylabel('Flux [a.u.]')
ax.grid(alpha=0.3)
ax.legend()
# ax.minorticks_on()
ax.tick_params(which='both', axis='both', direction='in', top=True, right=True)
fig.tight_layout()
plt.savefig('spectrum_slit_comparison.png', dpi=300)
plt.show()

In [ ]:
fig, ax = plt.subplots()
ax.imshow(images['img_035'])

In [9]:
plt.figure()
plt.plot(energy, exp_time[:,0])

In [22]:
plt.figure()
plt.plot(energy, max_intensity)